# Noise Models and Their Impact on SWAP Test Estimation

In this notebook, we explore how different types of quantum errors affect the accuracy of the SWAP test fidelity estimation. Understanding these errors is crucial for realistic shot planning and interpreting results from NISQ (Noisy Intermediate-Scale Quantum) devices.

## Sections
1. **Introduction to Quantum Noise**: Overview of error types.
2. **Error Models in Qiskit**: Setting up `NoiseModel`.
3. **Amplitude Damping (T1)**: Effect on SWAP test fidelity.
4. **Phase Damping (T2)**: Effect on interference.
5. **Depolarizing Error**: Fidelity underestimation.
6. **Combined Noise Models**: Realistic scenarios.
7. **Readout Errors**: Measurement misclassification.
8. **Error Mitigation Preview**: Intro to ZNE/PEC.
9. **Key Takeaways**: Practical adjustments.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from qiskit_aer.noise import (
    NoiseModel,
    depolarizing_error,
    amplitude_damping_error,
    phase_damping_error,
    ReadoutError,
    pauli_error
)

from qamp_shotplanner import (
    swap_test_1qubit,
    run_swap_fidelity_estimator,
    plan_shots_for_swap_fidelity
)

# Setup standard test case
theta1 = 0.3
theta2 = 0.8
qc = swap_test_1qubit(theta1, theta2)
F_ideal = math.cos((theta1 - theta2) / 2) ** 2

print(f"Test Case: |psi> = Ry({theta1}), |phi> = Ry({theta2})")
print(f"Ideal (noiseless) fidelity: {F_ideal:.4f}")

Test Case: |psi> = Ry(0.3), |phi> = Ry(0.8)
Ideal (noiseless) fidelity: 0.9388


## 1. Introduction to Quantum Noise

Quantum noise disrupts the delicate state of our qubits. For the SWAP test, which relies on interference to measure state overlap, noise generally leads to:
- **Bias**: The estimated fidelity is systematically lower (or different) than the true fidelity.
- **Variance**: Random errors can increase the spread of our estimates.

Common error channels include:
- **T1 (Amplitude Damping)**: Energy relaxation (|1> -> |0>).
- **T2 (Phase Damping)**: Loss of quantum phase coherence.
- **Depolarizing**: The state is replaced by a completely mixed state with some probability.
- **Readout**: Measuring '0' when the state is '1', and vice versa.

## 2. Error Models in Qiskit

Qiskit Aer provides a `NoiseModel` class to simulate these errors. We can attach specific errors to specific gates (like the Hadamard or CSWAP used in the SWAP test) or to all gates.

In [2]:
def evaluate_noise_model(noise_model, shots=10000, label="Noise Model"):
    """Helper to run the SWAP test under a specific noise model."""
    F_est = run_swap_fidelity_estimator(
        qc,
        shots=shots,
        noise_model=noise_model,
        seed_simulator=42
    )
    bias = F_est - F_ideal
    print(f"[{label}] F_est: {F_est:.4f}, Bias: {bias:.4f}")
    return F_est

## 3. Amplitude Damping (T1)

**Physics**: Qubits lose energy to the environment, relaxing from the excited state |1> to the ground state |0>.

**Impact**: If the SWAP test ancilla relaxes from |-> to |0> during the circuit (unlikely as it's mostly |+> or |0>), or if the data qubits relax, the interference is corrupted.

**Note**: Since `cswap` is a 3-qubit gate, we must construct a 3-qubit error channel (tensor product of 3 single-qubit T1 errors) to apply noise to it.

In [3]:
# Create an amplitude damping error
# param 'gamma' is prob of decay |1> -> |0>
gamma = 0.05 
error_t1 = amplitude_damping_error(gamma)

noise_t1 = NoiseModel()

# Apply to 1-qubit gates
noise_t1.add_all_qubit_quantum_error(error_t1, ["ry", "h"])

# Apply to 3-qubit cswap (tensor product of 3 single-qubit errors)
error_t1_3q = error_t1.tensor(error_t1).tensor(error_t1)
noise_t1.add_all_qubit_quantum_error(error_t1_3q, ["cswap"])

evaluate_noise_model(noise_t1, label="T1 Amplitude Damping")

[T1 Amplitude Damping] F_est: 0.8978, Bias: -0.0410


0.8977914222330206

## 4. Phase Damping (T2)

**Physics**: Loss of phase information without energy loss. The qubit state moves towards the Z-axis in the Bloch sphere.

**Impact**: Critical for SWAP test because it relies on the phase difference in the ancilla qubit to measure overlap. Dephasing destroys the interference pattern.

In [4]:
# Create a phase damping error
# param 'lighting' is prob of phase flip (simplified)
gamma_phase = 0.05
error_t2 = phase_damping_error(gamma_phase)

noise_t2 = NoiseModel()

# Apply to 1-qubit gates
noise_t2.add_all_qubit_quantum_error(error_t2, ["ry", "h"])

# Apply to 3-qubit cswap
error_t2_3q = error_t2.tensor(error_t2).tensor(error_t2)
noise_t2.add_all_qubit_quantum_error(error_t2_3q, ["cswap"])

evaluate_noise_model(noise_t2, label="T2 Phase Damping")

[T2 Phase Damping] F_est: 0.8867, Bias: -0.0521


0.886716916764473

## 5. Depolarizing Error

**Physics**: A "white noise" model. With probability $p$, the state is replaced by the maximally mixed state $I/2$. Equivalently, applying X, Y, or Z errors randomly.

**Impact**: This effectively "shrinks" the Bloch vector. For the SWAP test, it predictably lowers the measured overlap towards 0.5 (random guessing).

In [5]:
p_depol = 0.02
error_depol = depolarizing_error(p_depol, 1)

noise_depol = NoiseModel()
# Apply to single qubit gates
noise_depol.add_all_qubit_quantum_error(error_depol, ["ry", "h"])
# Apply 2-qubit error to CSWAP (often much higher in reality)
error_depol_2q = depolarizing_error(p_depol * 10, 3) # heuristic: 2q gates are noisier
noise_depol.add_all_qubit_quantum_error(error_depol_2q, ["cswap"])

evaluate_noise_model(noise_depol, label="Depolarizing")

[Depolarizing] F_est: 0.7079, Bias: -0.2309


0.7078806890237496

## 6. Combined Noise Models

Real devices suffer from all these errors simultaneously. We can construct a combined model.

In [6]:
# Combine T1, T2, and Depolarizing
# In Qiskit, we compose errors. 
# Note: This is a simplified manual construction. In practice, we often load from backend.

noise_combined = NoiseModel()
# We can just add them sequentially to the instruction
# But for simplicity here, let's use a composite error on the gates

error_composite = error_t1.compose(error_t2).compose(depolarizing_error(0.01, 1))
noise_combined.add_all_qubit_quantum_error(error_composite, ["ry", "h", "x", "z"])

evaluate_noise_model(noise_combined, label="Combined T1+T2+Depol")

[Combined T1+T2+Depol] F_est: 0.8736, Bias: -0.0652


0.8736217149583239

## 7. Readout Errors

**Physics**: The measurement apparatus misclassifies the state. 
- $P(0|1)$: Prob of measuring 0 when state is 1.
- $P(1|0)$: Prob of measuring 1 when state is 0.

**Impact**: Directly skews the statistics of the ancilla measurement.

In [7]:
p0given1 = 0.05
p1given0 = 0.02
readout_error = ReadoutError([[1 - p1given0, p1given0], [p0given1, 1 - p0given1]])

noise_readout = NoiseModel()
noise_readout.add_all_qubit_readout_error(readout_error)

evaluate_noise_model(noise_readout, label="Readout Error")

[Readout Error] F_est: 0.9388, Bias: -0.0000


0.9387912809451862

## 8. Error Mitigation Preview

Since noise introduces bias, simply increasing shots (reducing variance) is not enough. We need **Error Mitigation**.

1. **Zero-Noise Extrapolation (ZNE)**: Artificially increase the noise level and extrapolate back to zero noise.
2. **Probabilistic Error Cancellation (PEC)**: Learn the noise model and sample from an inverse channel to cancel it out (cost: more shots).
3. **Readout Error Mitigation (REM)**: Characterize the readout matrix and invert it classicaly.

These techniques trade *shots* (runtime) for *accuracy* (reduced bias). Our shot planner will need to account for this "unbiased variance overhead."

## 9. Key Takeaways

1. **Noise Bias vs. Shot Noise**: Shot planning (like Hoeffding) handles statistical uncertainty. It *does not* fix the systematic bias introduced by hardware noise.
2. **Conservative Planning**: In the presence of depolarizing noise, the measured fidelity shrinks. If we are trying to prove $F > F_{threshold}$, noise makes this harder, effectively requiring us to distinguish a smaller signal, which might require *more* shots or better hardware.
3. **Validation**: We must validate our expectations against the device's noise characteristics to trust our results.